# `shire.ipynb` — Reconciliation & QA walkthrough

This notebook demonstrates the two main analytical features of the `resume_rdf` library
using the fictional consultant CVs in `shire/` as test data:

| File | Person | Notes |
|------|--------|-------|
| `shire/frodo_baggins_cv.md` | Frodo Baggins | Senior management consultant, 6 projects |
| `shire/sam_gamgee_cv1.md` | Sam Gamgee | Data engineering consultant, chronological framing |
| `shire/sam_gamgee_cv2.md` | Sam Gamgee | Same career, ESG/sustainability framing |

Frodo and Sam share **3 projects** (Northern Energy Holdings, Mordor Industrial Group,
White Council Capital), described from different roles and perspectives.
Sam's two CV versions describe the same projects with different emphasis and wording —
making both a cross-person and a within-person deduplication test.

**Pre-requisite:** `pip install -e ".[all]"` and set `ANTHROPIC_API_KEY`.

In [ ]:
import os
import shutil
from pathlib import Path

# Set API key if not already in environment

SHIRE_DIR  = Path("shire") 
TTL_DIR    = SHIRE_DIR / "shire_ttl"
TTL_DIR.mkdir(exist_ok=True)

cvs = {
    "frodo":  SHIRE_DIR / "frodo_baggins_cv.md",
    "sam_v1": SHIRE_DIR / "sam_gamgee_cv1.md",
    "sam_v2": SHIRE_DIR / "sam_gamgee_cv2.md",
}
print("Input files:")
for name, p in cvs.items():
    print(f"  {name}: {p}  (exists: {p.exists()})")

---
## Section 1 — Entity Reconciliation

Goal: convert the three markdown CVs to Turtle RDF, then find and merge
entities (projects, employers) that refer to the same real-world thing
but carry slightly different names or IRIs across files.

### 1.1 — Convert markdown CVs to Turtle RDF

`generate_graph_from_file` calls the Claude API and returns a Turtle string.
Responses are cached in `cache/` (SHA-256 keyed), so re-running is free.

In [ ]:
from resume_rdf import generate_graph_from_file, count_triples

ttl_paths = {}
for name, cv_path in cvs.items():
    out_path = TTL_DIR / f"{name}.ttl"
    if out_path.exists():
        print(f"{name}: using cached {out_path}  ({count_triples(out_path.read_text())} triples)")
    else:
        print(f"{name}: generating from {cv_path} ...", end="", flush=True)
        turtle, usage = generate_graph_from_file(
            cv_path,
            extra_context="UK consultant CVs, energy and data engineering sectors. Output in English.",
            api_key = os.getenv("ANTHROPIC_API_KEY")
        )
        out_path.write_text(turtle)
        print(f"  {count_triples(turtle)} triples, {usage['output_tokens']:,} tokens")
    ttl_paths[name] = out_path

print("\nAll TTL files ready:")
for name, p in ttl_paths.items():
    print(f"  {name}: {p}")

### 1.2 — Extract labelled entities

`load_entities` parses each TTL file with rdflib and returns every
`cvx:Project` (labelled by `cvx:projectName`) and `cv:Company`
(labelled by `cv:Name`) node across all files.

In [ ]:
from resume_rdf import load_entities

entities = load_entities(list(ttl_paths.values()))
print(f"Extracted {len(entities)} entities total\n")

for kind in ("project", "company"):
    subset = [e for e in entities if e.kind == kind]
    print(f"{kind.upper()} ({len(subset)}):")
    for e in subset:
        print(f"  [{e.source.name}]  {e.label!r}")
    print()

### 1.3 — Find near-duplicate pairs

`find_matches` computes pairwise `difflib.SequenceMatcher` similarity
for every cross-file pair of the same entity type and returns those
above `threshold`.  We use 0.65 here to surface the shared-project
candidates even if the Claude parser worded them slightly differently.

In [ ]:
from resume_rdf import find_matches

matches = find_matches(entities, threshold=0.65)
print(f"Found {len(matches)} candidate pair(s)  (threshold=65%):\n")

for i, m in enumerate(matches, 1):
    print(f"[{i}]  {m.score.as_integer_ratio()}  {m.score:.0%}")
    print(f"     A: {m.a.label!r:50}  ({m.a.source.name})")
    print(f"     B: {m.b.label!r:50}  ({m.b.source.name})")
    print(f"     A IRI: {m.a.iri}")
    print(f"     B IRI: {m.b.iri}")
    print()

### 1.4 — Interactive reconciliation

`reconcile_interactive` presents each candidate pair at the terminal
and asks `[y/n/q(uit)]`.  It can't run interactively inside a notebook,
so the cell below shows what a typical session looks like.

To run it for real, use the CLI:
```bash
cv-reconcile shire_ttl/frodo.ttl shire_ttl/sam_v1.ttl shire_ttl/sam_v2.ttl --threshold 0.65
```

**Expected terminal session:**
```
Loading 3 file(s)…
  Extracted 28 entities (18 projects, 10 companies).

Found 6 candidate pair(s) at ≥65% similarity:

── [1/6]  PROJECT  (similarity 94%) ──
  A: 'Smart Grid Modernisation'            :proj_smartgrid_2022
     (frodo.ttl)
  B: 'Smart Grid Modernisation Programme'  :proj_smart_grid_modernisation
     (sam_v1.ttl)
  Canonical if merged → A  (:proj_smartgrid_2022)
  Same entity? [y/n/q(uit)] y
  ✓ Will rewrite  :proj_smart_grid_modernisation  →  :proj_smartgrid_2022

── [2/6]  PROJECT  (similarity 91%) ──
  A: 'Smart Grid Modernisation'            :proj_smartgrid_2022
     (frodo.ttl)
  B: 'Smart Grid Modernisation — Northern Energy Holdings'  :proj_smart_grid_neh
     (sam_v2.ttl)
  Canonical if merged → A  (:proj_smartgrid_2022)
  Same entity? [y/n/q(uit)] y
  ✓ Will rewrite  :proj_smart_grid_neh  →  :proj_smartgrid_2022

── [3/6]  PROJECT  (similarity 88%) ──
  A: 'Supply Chain Resilience Programme'   :proj_mordor_supply_chain
     (frodo.ttl)
  B: 'Supply Chain Resilience'             :proj_supply_chain
     (sam_v1.ttl)
  Same entity? [y/n/q(uit)] y

... (3 more pairs)

Applying 5 merge(s) across 3 file(s)…
  frodo.ttl:  0 triple(s) rewritten
  sam_v1.ttl: 12 triple(s) rewritten
  sam_v2.ttl: 17 triple(s) rewritten
Done.
```

### 1.5 — Apply reconciliation programmatically

For automated or batch use, call `apply_mapping` directly with
the `mapping` dict you build from confirmed matches.

Below we auto-accept all matches above threshold and apply them to
working copies of the TTL files, then verify the shared projects
now share a single IRI in the merged graph.

In [ ]:
from resume_rdf import apply_mapping, load_entities, find_matches

# Work on copies so the originals stay clean for re-use
rec_dir = SHIRE_DIR / Path("shire_reconciled")
if rec_dir.exists():
    shutil.rmtree(rec_dir)
shutil.copytree(TTL_DIR, rec_dir)
rec_paths = sorted(rec_dir.glob("*.ttl"))

# Build mapping from all matches (auto-yes)
rec_entities = load_entities(rec_paths)
rec_matches  = find_matches(rec_entities, threshold=0.65)

mapping = {}
for m in rec_matches:
    canon_a = mapping.get(m.a.iri, m.a.iri)
    canon_b = mapping.get(m.b.iri, m.b.iri)
    if canon_a != canon_b:
        mapping[canon_b] = canon_a

print(f"Applying {len(mapping)} IRI substitution(s):")
for old, new in mapping.items():
    print(f"  {str(old).rsplit('/', 1)[-1]}  →  {str(new).rsplit('/', 1)[-1]}")

results = apply_mapping(rec_paths, mapping)
print()
for path, count in sorted(results.items()):
    print(f"  {path.name}: {count} triple(s) rewritten")

In [ ]:
from rdflib import Graph, Namespace

_CVX = Namespace("http://example.org/cv-extension#")

# Load all reconciled files into one merged graph
merged = Graph()
for p in rec_paths:
    merged.parse(str(p), format="turtle")

print(f"Merged graph: {len(merged)} triples\n")
print("All cvx:Project IRIs in the merged graph:")

project_iris = set()
for proj_iri in merged.subjects(None, None):
    names = list(merged.objects(proj_iri, _CVX.projectName))
    if names:
        project_iris.add((str(proj_iri), str(names[0])))

for iri, name in sorted(project_iris, key=lambda x: x[1]):
    short = iri.rsplit("/", 1)[-1]
    print(f"  :{short:45}  {name!r}")

print("\nShared projects (IRI appears in > 1 source file):")
# Count how many source files contain each project IRI
from collections import Counter
iri_sources = Counter()
for p in rec_paths:
    g = Graph()
    g.parse(str(p), format="turtle")
    for iri, _, _ in g.triples((None, _CVX.projectName, None)):
        iri_sources[str(iri)] += 1

for iri, count in sorted(iri_sources.items(), key=lambda x: -x[1]):
    if count > 1:
        name = next(merged.objects(iri, _CVX.projectName), "?")
        short = iri.rsplit("/", 1)[-1]
        print(f"  :{short:45}  {name!r}  (appears in {count} files)")

---
## Section 2 — CV Audit & QA

Goal: inspect each Turtle CV for missing or incomplete fields and
demonstrate how to fill them in using `update_field`.

`audit_experience` checks every `cv:WorkHistory` and linked `cvx:Project`
for required predicates and returns a `Question` for each gap.

### 2.1 — Audit all three CVs

In [ ]:
from resume_rdf import audit_experience

for name, path in ttl_paths.items():
    questions = audit_experience(path)
    print(f"── {name}  ({path.name})  —  {len(questions)} question(s) ──")
    for q in questions:
        print(f"  [{q.slug}]  {q.field}")
        print(f"    → {q.question}")
    print()

### 2.2 — Inspect Frodo's questions as a DataFrame

In [ ]:
import pandas as pd

qs = audit_experience(ttl_paths["frodo"])
df = pd.DataFrame([
    {"slug": q.slug, "field": q.field, "question": q.question}
    for q in qs
])
df

### 2.3 — Fill in a missing value with `update_field`

`update_field(ttl_file, slug_or_iri, field, value)` resolves the node by
slug (the local part of its IRI), removes the old triple if present,
adds the new value with correct RDF datatype, and saves in-place.

We work on a copy so the cached original stays intact.

In [ ]:
from resume_rdf import update_field

qa_dir = SHIRE_DIR /Path("shire_qa")
qa_dir.mkdir(exist_ok=True)
frodo_qa = qa_dir / "frodo.ttl"
shutil.copy(ttl_paths["frodo"], frodo_qa)

# Baseline question count
before = audit_experience(frodo_qa)
print(f"Before: {len(before)} question(s)")
for q in before:
    print(f"  [{q.slug}]  {q.field}")

In [ ]:
# Answer the first open question
# In practice a user (or a conversational loop) would supply the real value;
# here we use plausible stand-ins keyed by field name.
sample_values = {
    "jobDescription":    "Led strategic consulting engagements for energy and financial services clients.",
    "endDate":           "2023-06-30",
    "startDate":         "2021-01-01",
    "jobTitle":          "Senior Consultant",
    "employedIn":        "http://example.org/cv/company_shire_consulting",
    "benefitsDelivered": "Delivered a 14% reduction in outage duration and secured 12,000 demand-response contracts.",
    "activitiesPerformed": "Managed programme governance, ran stakeholder workshops, and reviewed technical architecture.",
    "roleTitle":         "Programme Lead",
    "projectDescription":"Smart grid modernisation covering AMI rollout, SCADA integration, and demand-side response.",
    "usesSkill":         "(use update_field with the skill IRI — see note below)",
}

filled = []
for q in before:
    if q.field in sample_values and q.field != "usesSkill":
        value = sample_values[q.field]
        update_field(frodo_qa, q.slug, q.field, value)
        filled.append((q.slug, q.field, value))
        print(f"  set [{q.slug}].{q.field} = {value!r}")

print(f"\nFilled {len(filled)} field(s).")
print("Note: usesSkill requires a URIRef to an existing :skill_* node — handled separately.")

### 2.4 — Re-run the audit to confirm resolution

In [ ]:
after = audit_experience(frodo_qa)
print(f"Before: {len(before)} question(s)")
print(f"After:  {len(after)} question(s)")

remaining  = {(q.slug, q.field) for q in after}
resolved   = [(q.slug, q.field) for q in before if (q.slug, q.field) not in remaining]
still_open = [(q.slug, q.field) for q in after]

print(f"\nResolved ({len(resolved)}):")
for slug, field in resolved:
    print(f"  ✓ [{slug}]  {field}")

if still_open:
    print(f"\nStill open ({len(still_open)}):")
    for slug, field in still_open:
        print(f"  ? [{slug}]  {field}")

### 2.5 — Compare Sam v1 vs v2 audit profiles

The two versions of Sam's CV will likely produce different question sets —
v2 (ESG framing) tends to be more verbose in project descriptions, so it
may have fewer `projectDescription` and `benefitsDelivered` gaps.
This cell shows the field-level diff.

In [ ]:
q_v1 = {(q.slug, q.field) for q in audit_experience(ttl_paths["sam_v1"])}
q_v2 = {(q.slug, q.field) for q in audit_experience(ttl_paths["sam_v2"])}

only_in_v1 = q_v1 - q_v2
only_in_v2 = q_v2 - q_v1
in_both    = q_v1 & q_v2

print(f"Gaps in v1 only ({len(only_in_v1)} — filled by v2 phrasing):")
for slug, field in sorted(only_in_v1):
    print(f"  [{slug}]  {field}")

print(f"\nGaps in v2 only ({len(only_in_v2)} — filled by v1 phrasing):")
for slug, field in sorted(only_in_v2):
    print(f"  [{slug}]  {field}")

print(f"\nGaps in both versions ({len(in_both)}):")
for slug, field in sorted(in_both):
    print(f"  [{slug}]  {field}")

### 2.6 — Export questions as JSON (for downstream tooling)

The `Question` dataclass is a plain frozen dataclass — easy to serialise.
This is the shape expected by any conversational loop that picks up the
questions and presents them to the CV owner.

In [ ]:
import json
from resume_rdf import Question

all_questions = {}
for name, path in ttl_paths.items():
    qs = audit_experience(path)
    all_questions[name] = [
        {"slug": q.slug, "field": q.field, "question": q.question}
        for q in qs
    ]

out_path = SHIRE_DIR / Path("shire_qa") / "questions.json"
out_path.parent.mkdir(exist_ok=True)
out_path.write_text(json.dumps(all_questions, indent=2))
print(f"Wrote {sum(len(v) for v in all_questions.values())} questions to {out_path}")
print()
print(json.dumps(all_questions, indent=2)[:800], "...")

---
## Section 3 — Graph Visualisation

Goal: render each Turtle CV as an interactive HTML network graph (pyvis) or a
static image (networkx + matplotlib).  

Nodes are colour-coded by type:

| Colour | Shape | Node type |
|--------|-------|-----------|
| blue  | ★ star   | Person |
| orange | ■ square | Company / employer |
| green  | ◆ diamond | Project |
| brown  | ● dot    | Skill |
| yellow | ■ square | Education |
| grey   | ● dot    | Training / MOOC |
| red    | ◆ diamond | Personal project |
| purple | ● dot    | Publication |

Edges carry the role label (job title or "uses").

In [ ]:
# requires: pip install "resume-rdf[viz]"
from resume_rdf import visualize_cv
from pathlib import Path
print("visualize_cv imported OK")

### 3.1 — Interactive HTML graph (Frodo)

Pass any `.html` output path and `visualize_cv` uses pyvis with Barnes-Hut
physics for a force-directed, zoomable graph.  The file is self-contained
(no server needed — open directly in a browser).

In [ ]:
html_path = visualize_cv(
    "shire/shire_ttl/frodo.ttl",
    "shire/shire_ttl/frodo_graph.html",
)
print(f"Interactive graph → {html_path}")

### 3.2 — Display inline in the notebook

The `IFrame` widget embeds the pyvis HTML directly.  Use `width="100%"` and
a comfortable `height` — 620 px works well for most CVs.

In [ ]:
from IPython.display import IFrame
IFrame(src=str(html_path), width="100%", height=620)

### 3.3 — Static PNG (Sam v1)

Pass a `.png` (or `.svg` / `.pdf`) output path and `visualize_cv` switches to
networkx + matplotlib.  A shell layout places the person in the centre, employers
in the first ring, projects in the second, and skills/education/other in the outer ring.

In [ ]:
png_path = visualize_cv(
    "shire/shire_ttl/sam_v1.ttl",
    "shire/shire_ttl/sam_v1_graph.png",
)
from IPython.display import Image
Image(str(png_path), width=800)

### 3.4 — Side-by-side comparison: Sam v1 vs Sam v2

Generating both PNGs and placing them in matplotlib subplots makes it easy to
see how the two framings of the same career differ structurally — v2 (ESG focus)
typically produces more skill nodes and richer project edges.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

visualize_cv("shire/shire_ttl/sam_v1.ttl", "shire/shire_ttl/sam_v1_graph.png")
visualize_cv("shire/shire_ttl/sam_v2.ttl", "shire/shire_ttl/sam_v2_graph.png")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, path, title in zip(
    axes,
    ["shire/shire_ttl/sam_v1_graph.png", "shire/shire_ttl/sam_v2_graph.png"],
    ["Sam v1 — chronological", "Sam v2 — ESG framing"],
):
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=13)
    ax.axis("off")
plt.tight_layout()
plt.show()

### CLI equivalent

```bash
cv-graph shire/shire_ttl/frodo.ttl --output shire/shire_ttl/frodo_graph.html
cv-graph shire/shire_ttl/sam_v1.ttl --output shire/shire_ttl/sam_v1_graph.png
cv-graph shire/shire_ttl/sam_v2.ttl --output shire/shire_ttl/sam_v2_graph.svg
```

---
## Section 4 — TTL → Markdown Export

`ttl_to_markdown` inverts the pipeline: it reads a Turtle file and reconstructs
a clean, human-readable Markdown CV.  Sections produced:

- **Header** — name, contact, summary
- **Skills** — with level and years of experience
- **Work history** — employer, dates, role; nested projects with activities, outcomes, skills
- **Education** — institution, degree, dates
- **Training / MOOCs**
- **Personal projects**
- **Publications**

The output is useful for reviewing exactly what was captured, generating a cleaned CV,
or comparing the structured representation against the raw source.

In [ ]:
# requires: pip install "resume-rdf[export]"
from resume_rdf import ttl_to_markdown
print("ttl_to_markdown imported OK")

### 4.1 — Convert all three CVs

In [ ]:
RECO_DIR  = Path("shire/reconstructed/") 
RECO_DIR.mkdir(exist_ok=True)

for name in ["frodo", "sam_v1", "sam_v2"]:
    md_text = ttl_to_markdown(f"shire/shire_ttl/{name}.ttl")
    out = Path(f"shire/reconstructed/{name}_reconstructed.md")
    out.write_text(md_text)
    print(f"Wrote {out}  ({len(md_text):,} chars)")

### 4.2 — Print Frodo's reconstructed CV

The output shows exactly what the graph captured — useful as a quick sanity check
after generation or after patching fields with .

In [ ]:
print(ttl_to_markdown("shire/shire_ttl/frodo.ttl"))

### 4.3 — Round-trip diff: original vs reconstructed

The diff reveals two things:

- **Information loss** — free-form prose that the parser discarded or
  condensed into a single literal
- **Structural gain** — typed dates, explicit skill links, and section
  headings that the raw Markdown lacked

We print the first 60 lines to keep the output manageable.

In [ ]:
import difflib

original      = Path("shire/frodo_baggins_cv.md").read_text()
reconstructed = (RECO_DIR / Path("frodo_reconstructed.md")).read_text()

diff = list(difflib.unified_diff(
    original.splitlines(),
    reconstructed.splitlines(),
    fromfile="original",
    tofile="reconstructed",
    lineterm="",
))
print(f"Total diff lines: {len(diff)}")
print()
print("".join(diff[:60]))

### CLI equivalent

```bash
# print to stdout
cv-to-md shire/shire_ttl/frodo.ttl

# write to file
cv-to-md shire/shire_ttl/sam_v1.ttl --output shire/reconstructed/sam_v1_reconstructed.md
cv-to-md shire/shire_ttl/sam_v2.ttl --output shire/reconstructed/sam_v2_reconstructed.md
```

---
## Section 5 — TTL Consolidation (same-person merge)

Goal: Sam Gamgee has two CV framings — `sam_v1` (chronological) and `sam_v2`
(ESG focus).  They describe the same projects with different wording and
different levels of detail.  `consolidate_ttls` merges them into one enriched,
deduplicated TTL with no information loss.

Merge heuristics applied per `(subject, predicate)` conflict:

| Predicate type | Strategy |
|----------------|----------|
| URI-valued (skills, type links) | **Union** — all values kept |
| Description / title literals | **Longest string** wins |
| `startDate` | **Earliest** date wins |
| `endDate` | `"present"` beats any date; otherwise **latest** |
| Other literals | **Longest string** wins |

In [ ]:
# requires: pip install "resume-rdf[merge]"
from resume_rdf import consolidate_ttls, MergeStats
print("consolidate_ttls imported OK")

### 5.1 — Merge Sam v1 + Sam v2

`consolidate_ttls` first reconciles entity IRIs across the two files
(rewriting project/company IRIs to a single canonical form), then merges
all triples and resolves literal conflicts with the heuristics above.

In [ ]:
merge_dir = SHIRE_DIR / "shire_merged"
merge_dir.mkdir(exist_ok=True)
sam_merged = merge_dir / "sam_merged.ttl"

stats = consolidate_ttls(
    [ttl_paths["sam_v1"], ttl_paths["sam_v2"]],
    sam_merged,
    threshold=0.70,
)

print(f"Input files      : {stats.input_files}")
print(f"Input triples    : {stats.input_triples}")
print(f"IRI mappings     : {stats.iri_mappings}")
print(f"Conflicts resolved: {stats.conflicts_resolved}")
print(f"Output triples   : {stats.output_triples}")
print(f"Saved to         : {sam_merged}")

### 5.2 — Inspect the merged graph

Confirm that the shared projects (Northern Energy, Mordor, White Council)
now appear exactly once and carry the fuller descriptions from whichever
version was more detailed.

In [ ]:
from rdflib import Graph, Namespace

_CVX = Namespace("http://example.org/cv-extension#")
_CV  = Namespace("http://purl.org/captsolo/resume-rdf/0.2/cv#")

g_merged = Graph()
g_merged.parse(str(sam_merged), format="turtle")

print(f"Merged graph: {len(g_merged)} triples\n")
print("Projects in merged graph:")
for proj in sorted(g_merged.subjects(_CVX.projectName, None)):
    name = next(g_merged.objects(proj, _CVX.projectName), "")
    desc = next(g_merged.objects(proj, _CVX.projectDescription), "(no description)")
    short = str(proj).rsplit("/", 1)[-1]
    print(f"  :{short}")
    print(f"    name: {name}")
    print(f"    desc: {str(desc)[:100]}...")
    print()

### 5.3 — Before / after triple counts

The merged graph should have significantly fewer triples than the naive
sum of both inputs (because duplicate nodes are merged), but more unique
information per node than either input alone.

In [ ]:
from resume_rdf import count_triples

n_v1 = count_triples(ttl_paths["sam_v1"].read_text())
n_v2 = count_triples(ttl_paths["sam_v2"].read_text())
n_merged = len(g_merged)

print(f"sam_v1 triples   : {n_v1}")
print(f"sam_v2 triples   : {n_v2}")
print(f"sum (naive)      : {n_v1 + n_v2}")
print(f"merged triples   : {n_merged}")
print(f"dedup ratio      : {n_merged / (n_v1 + n_v2):.0%}")

### 5.4 — Visualise the merged graph

The merged TTL can be passed directly to `visualize_cv` — the resulting
graph shows all projects from both framings in a single view.

In [ ]:
from resume_rdf import visualize_cv
from IPython.display import IFrame

merged_html = visualize_cv(sam_merged, merge_dir / "sam_merged_graph.html")
IFrame(src=str(merged_html), width="100%", height=620)

### 5.5 — Export merged CV to Markdown

The consolidated TTL can also be exported to Markdown — the result is a
single CV that combines the best descriptions from both versions.

In [ ]:
from resume_rdf import ttl_to_markdown

merged_md = ttl_to_markdown(sam_merged)
out_path = merge_dir / "sam_merged_cv.md"
out_path.write_text(merged_md)
print(f"Wrote {out_path}  ({len(merged_md):,} chars)\n")
print(merged_md[:1500], "...")

### CLI equivalent

```bash
cv-merge shire/shire_ttl/sam_v1.ttl shire/shire_ttl/sam_v2.ttl \
         --output shire/shire_merged/sam_merged.ttl

# Then visualise and export the merged file:
cv-graph  shire/shire_merged/sam_merged.ttl --output shire/shire_merged/sam_merged_graph.html
cv-to-md  shire/shire_merged/sam_merged.ttl --output shire/shire_merged/sam_merged_cv.md
```

### 5.6 — Compare all three merge strategies

Run `consolidate_ttls` three times (one per strategy) and compare how the
same conflicting `projectDescription` field is resolved in each case.

| Strategy | Behaviour | Trade-off |
|----------|-----------|----------|
| `longest` | Keep longest string | Fast; may silently lose info |
| `concat` | Join all values with `" | "` | Zero loss; can be verbose |
| `llm` | Claude synthesises one coherent text | Best quality; API cost (cached) |

In [ ]:
from resume_rdf import consolidate_ttls
from rdflib import Graph, Namespace

_CVX = Namespace("http://example.org/cv-extension#")

strategies = ["longest", "concat", "llm"]
results = {}

for strat in strategies:
    out = merge_dir / f"sam_merged_{strat}.ttl"
    stats = consolidate_ttls(
        [ttl_paths["sam_v1"], ttl_paths["sam_v2"]],
        out,
        strategy=strat,
        # api_key="sk-ant-..."  # or set ANTHROPIC_API_KEY
    )
    g = Graph()
    g.parse(str(out), format="turtle")
    results[strat] = (stats, g)
    suffix = ""
    if strat == "llm":
        suffix = f"  (API calls: {stats.llm_calls}, cache hits: {stats.llm_cache_hits})"
    line = f"[{strat:7}]  {stats.output_triples} triples, {stats.conflicts_resolved} conflicts resolved"
    print(line + suffix)

### 5.7 — Side-by-side description comparison

Pick the Northern Energy Smart Grid project (shared across both CVs)
and print how each strategy resolved the `projectDescription` conflict.

In [ ]:
# Find the canonical IRI for the Northern Energy project in the merged graph
g_ref = results["longest"][1]
target_iri = None
for proj in g_ref.subjects(_CVX.projectName, None):
    name = str(next(g_ref.objects(proj, _CVX.projectName), ""))
    if "Northern Energy" in name or "Smart Grid" in name:
        target_iri = proj
        break

if target_iri is None:
    print("Project not found — adjust the name filter above")
else:
    print(f"Project IRI: {target_iri}\n")
    for strat in strategies:
        g = results[strat][1]
        desc = next(g.objects(target_iri, _CVX.projectDescription), "(not found)")
        print(f"--- {strat} ---")
        print(str(desc))
        print()

### 5.8 — Triple count comparison

`concat` and `llm` produce the same triple count as `longest`
(one triple per subject+predicate pair), but the literal values differ.

In [ ]:
import pandas as pd

rows = []
for strat, (stats, _) in results.items():
    rows.append({
        "strategy": strat,
        "input_triples": stats.input_triples,
        "iri_mappings": stats.iri_mappings,
        "conflicts_resolved": stats.conflicts_resolved,
        "output_triples": stats.output_triples,
        "llm_calls": stats.llm_calls,
        "llm_cache_hits": stats.llm_cache_hits,
    })

df = pd.DataFrame(rows).set_index("strategy")
df

---
## Summary

| Feature | Function | CLI |
|---------|----------|-----|
| CV to Turtle RDF | `generate_graph_from_file` / `generate_graph_from_bytes` | `cv-to-rdf` |
| Extract entities | `load_entities(ttl_files)` | -- |
| Find near-duplicate pairs | `find_matches(entities, threshold)` | -- |
| Interactive reconciliation | `reconcile_interactive(ttl_files)` | `cv-reconcile` |
| Apply IRI mapping | `apply_mapping(ttl_files, mapping)` | -- |
| Audit for missing fields | `audit_experience(ttl_file)` | `cv-audit` |
| Update a field | `update_field(ttl_file, slug, field, value)` | `cv-update` |
| Visualise graph | `visualize_cv(ttl_file, output_path)` | `cv-graph` |
| Export to Markdown | `ttl_to_markdown(ttl_file)` | `cv-to-md` |
| Consolidate same-person TTLs | `consolidate_ttls(ttl_files, output)` | `cv-merge` |

The `shire/` test CVs are designed so that:
- **Cross-person** reconciliation: 3 shared projects appear in Frodo and Sam's files
- **Within-person** reconciliation: Sam's v1 and v2 describe the same projects differently
- **Audit coverage**: both CVs will produce at least some open questions
- **Visualisation**: each CV maps to a distinct subgraph topology
- **Round-trip**: the diff between original Markdown and reconstructed output surfaces
  what the graph representation preserves and what it discards
- **Consolidation**: Sam v1 + v2 merge into one enriched TTL with no information loss